# Prompt Engineering: In-Context Learning and Structural Anatomy

**Session:** 2026-11-09 · Text Project: ChatGPT-based Application  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking  

---

## 1. Mechanics of In-Context Learning (ICL)

In-Context Learning (ICL) refers to the capability of pre-trained language models to condition on prompt demonstrations and solve novel tasks at inference time without explicit parameter updates or backpropagation:

$$\theta^* = \theta_0 \quad (\text{parameters remain fixed})$$

Research reveals that ICL operates via **induction heads** in intermediate Transformer attention layers, performing implicit Bayesian inference or meta-gradient descent within the internal activation space.

### The Four Pillars of a Production Prompt
A robust prompt consists of four decoupled structural components:
1. **Task Description (Directive):** High-level instruction specifying the intended transformation or classification.
2. **Context / General Setting:** Background domain constraints, reference materials, or persona grounding.
3. **Current Input Data (Delimited):** Target data payload isolated using strict delimiters (e.g. `"""`, XML tags, or markdown backticks) to prevent prompt injection.
4. **Output Indicator / Schema:** Formatting constraints guiding the exact syntax of the completion.


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path


@dataclass
class PromptTemplate:
    task_description: str
    context: str | None = None
    output_indicator: str | None = None

    def format(self, input_text: str, examples: list[dict[str, str]] | None = None) -> str:
        parts = [f"TASK: {self.task_description}"]
        if self.context:
            parts.append(f"CONTEXT: {self.context}")
        if examples:
            parts.append("EXAMPLES:")
            for i, ex in enumerate(examples, 1):
                parts.append(
                    f"  Example {i}:\n    Input: {ex['input']}\n    Output: {ex['output']}"
                )
        parts.append(f"INPUT (Delimited):\n---\n{input_text}\n---")
        if self.output_indicator:
            parts.append(f"OUTPUT: {self.output_indicator}")
        return "\n\n".join(parts)


template = PromptTemplate(
    task_description="Classify the sentiment of the provided customer review into [positive, neutral, negative].",
    output_indicator="Return a single label followed by a one-sentence rationale.",
)

sample_prompt = template.format("The delivery was fast, but the keyboard keys feel slightly loose.")
print(sample_prompt)

## 2. In-Context Learning Progression: Zero-Shot, One-Shot, and Few-Shot

Language models improve prediction accuracy, schema compliance, and boundary calibration as prompt demonstrations increase from Zero-Shot to Few-Shot:
- **Zero-Shot:** Pure instruction without exemplars. Depends entirely on the model's pre-trained semantic priors.
- **One-Shot:** A single demonstration anchor providing formatting and decision calibration.
- **Few-Shot ($k$-Shot):** Multiple balanced demonstrations ($k \ge 2$). Establishes label distribution priors and edge-case handling.


In [ ]:
DATA_PATH = Path("../data/sentiment_classification_benchmarks.json")
with DATA_PATH.open(encoding="utf-8") as f:
    benchmarks = json.load(f)

print(f"Loaded {len(benchmarks)} benchmark instances from dataset.")

# Select few-shot demonstrations
few_shot_exemplars = [
    {"input": benchmarks[1]["text"], "output": benchmarks[1]["label"]},  # 'So bad.' -> negative
    {
        "input": benchmarks[2]["text"],
        "output": benchmarks[2]["label"],
    },  # 'I loved this movie!' -> positive
    {
        "input": benchmarks[0]["text"],
        "output": benchmarks[0]["label"],
    },  # 'I think the food was okay.' -> neutral
]

# Zero-Shot Prompt
zero_shot_prompt = template.format(benchmarks[4]["text"])

# Few-Shot Prompt
few_shot_prompt = template.format(benchmarks[4]["text"], examples=few_shot_exemplars)

print("=== ZERO-SHOT PROMPT ===")
print(zero_shot_prompt)
print("\n=== FEW-SHOT PROMPT ===")
print(few_shot_prompt)

## 3. Structured Output Extraction & JSON Schema Parsing

Production pipelines require predictable JSON outputs rather than free-form conversational prose. Below, we implement a defensive parser that validates and cleans model outputs into strongly-typed Python dictionaries.


In [ ]:
import re


def parse_structured_output(raw_output: str) -> dict[str, str]:
    """Extract and validate structured JSON-like key-value responses."""
    # Strip markdown codeblocks if present
    code_match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", raw_output)
    clean_text = code_match.group(1) if code_match else raw_output.strip()

    try:
        return json.loads(clean_text)
    except json.JSONDecodeError:
        # Fallback regex parsing for line-oriented outputs
        extracted = {}
        for line in clean_text.splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                extracted[k.strip().lower()] = v.strip()
        return extracted


# Simulated model response containing structured JSON
mock_model_response = """```json
{
  "label": "positive",
  "confidence": 0.94,
  "explanation": "The reviewer expressed explicit enjoyment using the phrase 'loved this movie'."
}
```"""

parsed = parse_structured_output(mock_model_response)
print("Parsed Structured Output:")
for key, value in parsed.items():
    print(f"  {key}: {value}")

assert parsed["label"] == "positive"
assert parsed["confidence"] == 0.94
print("\nJSON Schema parsing verified successfully.")